# Filtering with WHERE

## Overview
`WHERE` is applied after `FROM` but before `SELECT` — it determines which rows enter the rest of the query. Any expression that evaluates to TRUE, FALSE, or NULL can appear in WHERE. Only TRUE rows pass through; NULL rows are silently excluded.

**Operator reference:**

| Category | Syntax | Notes |
|---|---|---|
| Equality | `col = val` / `col <> val` | `!=` is equivalent to `<>` |
| Comparison | `<` `>` `<=` `>=` | Works on numbers, dates (ISO text), text |
| Range | `BETWEEN a AND b` | Inclusive on both ends |
| Set | `IN (v1, v2, ...)` / `NOT IN (...)` | NOT IN with NULLs in list is a trap (see pitfalls) |
| Pattern | `LIKE 'pat%'` / `NOT LIKE` | `%` = any chars, `_` = exactly one char; case rules vary by DB |
| Null | `IS NULL` / `IS NOT NULL` | Never use `= NULL` |
| Existence | `EXISTS (subquery)` | Covered in depth in `05_subqueries_ctes` |
| Logical | `AND` `OR` `NOT` | AND binds tighter than OR |

**NULL propagation in WHERE:** `NULL AND TRUE` → NULL (row excluded). `NULL OR TRUE` → TRUE (row included). Understanding this prevents subtle bugs in complex conditions.

---

In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

cur.executescript("""
CREATE TABLE encounters (
    enc_id      INTEGER PRIMARY KEY,
    patient_id  INTEGER,
    enc_date    TEXT,
    dept        TEXT,
    diag_code   TEXT,       -- ICD-10 code; NULL if not yet coded
    cost_cad    REAL,
    admitted    INTEGER,    -- 1 = inpatient admission, 0 = outpatient
    provider_id INTEGER
);

INSERT INTO encounters VALUES
  (1,  1,  '2023-01-15', 'Emergency',    'J18.9',  1850.00, 1, 10),
  (2,  2,  '2023-02-20', 'Cardiology',   'I25.1',  3200.00, 1, 11),
  (3,  3,  '2023-03-05', 'Outpatient',   'Z00.0',   120.00, 0, 12),
  (4,  4,  '2023-03-18', 'Orthopaedics', 'M17.1',  5500.00, 1, 13),
  (5,  5,  '2023-04-02', 'Outpatient',   'J06.9',    95.00, 0, 12),
  (6,  6,  '2023-05-11', 'Emergency',    'R07.9',   780.00, 0, 10),
  (7,  7,  '2023-06-22', 'Cardiology',   'I10',    2100.00, 1, 11),
  (8,  8,  '2023-07-14', 'Outpatient',    NULL,      80.00, 0, 12),
  (9,  1,  '2023-08-30', 'Emergency',    'R55',     410.00, 0, 10),
  (10, 3,  '2023-09-12', 'Outpatient',   'Z00.0',   110.00, 0, 12),
  (11, 9,  '2023-10-01', 'Cardiology',   'I48.0',  1750.00, 1, 11),
  (12, 10, '2023-11-03', 'Emergency',    'S52.5',  2200.00, 1, 10),
  (13, 2,  '2023-11-18', 'Outpatient',    NULL,      90.00, 0, 12),
  (14, 5,  '2023-12-05', 'Orthopaedics', 'M54.5',   450.00, 0, 13);
""")
conn.commit()

print('encounters table ready —', cur.execute('SELECT COUNT(*) FROM encounters').fetchone()[0], 'rows')
print()
print(pd.read_sql('SELECT * FROM encounters', conn).to_string(index=False))


encounters table ready — 14 rows

 enc_id  patient_id   enc_date         dept diag_code  cost_cad  admitted  provider_id
      1           1 2023-01-15    Emergency     J18.9    1850.0         1           10
      2           2 2023-02-20   Cardiology     I25.1    3200.0         1           11
      3           3 2023-03-05   Outpatient     Z00.0     120.0         0           12
      4           4 2023-03-18 Orthopaedics     M17.1    5500.0         1           13
      5           5 2023-04-02   Outpatient     J06.9      95.0         0           12
      6           6 2023-05-11    Emergency     R07.9     780.0         0           10
      7           7 2023-06-22   Cardiology       I10    2100.0         1           11
      8           8 2023-07-14   Outpatient      None      80.0         0           12
      9           1 2023-08-30    Emergency       R55     410.0         0           10
     10           3 2023-09-12   Outpatient     Z00.0     110.0         0           12
     11  

---
## Comparison and range operators


In [7]:
# Simple comparison
print('=== High-cost encounters (cost > $1000) ===')
q = """
SELECT enc_id, dept, diag_code, cost_cad, admitted
FROM   encounters
WHERE  cost_cad > 1000
ORDER BY cost_cad DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# BETWEEN — inclusive on both ends
print()
print('=== Encounters in Q1 2023 (BETWEEN) ===')
q2 = """
SELECT enc_id, patient_id, enc_date, dept, cost_cad
FROM   encounters
WHERE  enc_date BETWEEN '2023-01-01' AND '2023-03-31'
ORDER BY enc_date
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# NOT BETWEEN
print()
print('=== Mid-range cost $200 to $1999 ===')
q3 = """
SELECT enc_id, dept, cost_cad
FROM   encounters
WHERE  cost_cad BETWEEN 200 AND 1999
ORDER BY cost_cad
"""
print(pd.read_sql(q3, conn).to_string(index=False))


=== High-cost encounters (cost > $1000) ===
 enc_id         dept diag_code  cost_cad  admitted
      4 Orthopaedics     M17.1    5500.0         1
      2   Cardiology     I25.1    3200.0         1
     12    Emergency     S52.5    2200.0         1
      7   Cardiology       I10    2100.0         1
      1    Emergency     J18.9    1850.0         1
     11   Cardiology     I48.0    1750.0         1

=== Encounters in Q1 2023 (BETWEEN) ===
 enc_id  patient_id   enc_date         dept  cost_cad
      1           1 2023-01-15    Emergency    1850.0
      2           2 2023-02-20   Cardiology    3200.0
      3           3 2023-03-05   Outpatient     120.0
      4           4 2023-03-18 Orthopaedics    5500.0

=== Mid-range cost $200 to $1999 ===
 enc_id         dept  cost_cad
      9    Emergency     410.0
     14 Orthopaedics     450.0
      6    Emergency     780.0
     11   Cardiology    1750.0
      1    Emergency    1850.0


---
## IN, NOT IN, and LIKE


In [8]:
# IN for set membership
print('=== Emergency or Cardiology encounters ===')
q = """
SELECT enc_id, dept, cost_cad, admitted
FROM   encounters
WHERE  dept IN ('Emergency', 'Cardiology')
ORDER BY dept, cost_cad DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# NOT IN — important: fails silently if list contains NULL (see pitfalls)
print()
print('=== Non-outpatient encounters (NOT IN) ===')
q2 = """
SELECT enc_id, dept, cost_cad
FROM   encounters
WHERE  dept NOT IN ('Outpatient')
ORDER BY enc_date
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# LIKE — prefix match (can use index), suffix match (cannot)
print()
print('=== Respiratory diagnoses (ICD codes starting with J) ===')
q3 = """
SELECT enc_id, diag_code, dept, cost_cad
FROM   encounters
WHERE  diag_code LIKE 'J%'
"""
print(pd.read_sql(q3, conn).to_string(index=False))

print()
print('=== Any diagnosis containing 5 (single-char wildcard demo) ===')
q4 = """
SELECT enc_id, diag_code
FROM   encounters
WHERE  diag_code LIKE '%5_'   -- a '5' followed by exactly one more char
"""
print(pd.read_sql(q4, conn).to_string(index=False))


=== Emergency or Cardiology encounters ===
 enc_id       dept  cost_cad  admitted
      2 Cardiology    3200.0         1
      7 Cardiology    2100.0         1
     11 Cardiology    1750.0         1
     12  Emergency    2200.0         1
      1  Emergency    1850.0         1
      6  Emergency     780.0         0
      9  Emergency     410.0         0

=== Non-outpatient encounters (NOT IN) ===
 enc_id         dept  cost_cad
      1    Emergency    1850.0
      2   Cardiology    3200.0
      4 Orthopaedics    5500.0
      6    Emergency     780.0
      7   Cardiology    2100.0
      9    Emergency     410.0
     11   Cardiology    1750.0
     12    Emergency    2200.0
     14 Orthopaedics     450.0

=== Respiratory diagnoses (ICD codes starting with J) ===
 enc_id diag_code       dept  cost_cad
      1     J18.9  Emergency    1850.0
      5     J06.9 Outpatient      95.0

=== Any diagnosis containing 5 (single-char wildcard demo) ===
 enc_id diag_code
      9       R55


---
## NULL checks: IS NULL and IS NOT NULL


In [9]:
# Encounters with no diagnosis code yet
print('=== Encounters with missing diagnosis code (IS NULL) ===')
q = """
SELECT enc_id, patient_id, enc_date, dept, cost_cad
FROM   encounters
WHERE  diag_code IS NULL
"""
print(pd.read_sql(q, conn).to_string(index=False))

# What = NULL does (wrong — always returns nothing)
print()
print('=== Wrong: WHERE diag_code = NULL (always returns 0 rows) ===')
wrong = pd.read_sql("SELECT COUNT(*) AS n FROM encounters WHERE diag_code = NULL", conn)
print(wrong.to_string(index=False))

# NULL with AND/OR
print()
print('=== NULL OR TRUE = TRUE (row is included) ===')
# A NULL diag_code OR admitted=1: admitted=1 rows appear even when diag is NULL
q2 = """
SELECT enc_id, diag_code, admitted
FROM   encounters
WHERE  diag_code LIKE 'I%' OR admitted = 1
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Encounters with missing diagnosis code (IS NULL) ===
 enc_id  patient_id   enc_date       dept  cost_cad
      8           8 2023-07-14 Outpatient      80.0
     13           2 2023-11-18 Outpatient      90.0

=== Wrong: WHERE diag_code = NULL (always returns 0 rows) ===
 n
 0

=== NULL OR TRUE = TRUE (row is included) ===
 enc_id diag_code  admitted
      1     J18.9         1
      2     I25.1         1
      4     M17.1         1
      7       I10         1
     11     I48.0         1
     12     S52.5         1


---
## AND, OR, and operator precedence


In [10]:
# AND binds tighter than OR — use parentheses to be explicit
print('=== Admitted Emergency or high-cost Cardiology ===')
# Without parens: admitted=1 AND dept='Emergency' OR cost>2000 AND dept='Cardiology'
# That is already unambiguous here, but let's be explicit:
q = """
SELECT enc_id, dept, cost_cad, admitted
FROM   encounters
WHERE  (admitted = 1 AND dept = 'Emergency')
    OR (cost_cad > 2000 AND dept = 'Cardiology')
ORDER BY dept, cost_cad DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# NOT
print()
print('=== Encounters that are NOT outpatient ===')
q2 = """
SELECT enc_id, dept, cost_cad
FROM   encounters
WHERE  NOT dept = 'Outpatient'
ORDER BY enc_date
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# Complex real-world filter: admitted cardiology/emergency with diagnosis, cost > 500
print()
print('=== Admitted encounters, cardiac/emergency depts, coded, cost > $500 ===')
q3 = """
SELECT enc_id, patient_id, enc_date, dept, diag_code, cost_cad
FROM   encounters
WHERE  admitted = 1
  AND  dept IN ('Cardiology', 'Emergency')
  AND  diag_code IS NOT NULL
  AND  cost_cad > 500
ORDER BY cost_cad DESC
"""
print(pd.read_sql(q3, conn).to_string(index=False))


=== Admitted Emergency or high-cost Cardiology ===
 enc_id       dept  cost_cad  admitted
      2 Cardiology    3200.0         1
      7 Cardiology    2100.0         1
     12  Emergency    2200.0         1
      1  Emergency    1850.0         1

=== Encounters that are NOT outpatient ===
 enc_id         dept  cost_cad
      1    Emergency    1850.0
      2   Cardiology    3200.0
      4 Orthopaedics    5500.0
      6    Emergency     780.0
      7   Cardiology    2100.0
      9    Emergency     410.0
     11   Cardiology    1750.0
     12    Emergency    2200.0
     14 Orthopaedics     450.0

=== Admitted encounters, cardiac/emergency depts, coded, cost > $500 ===
 enc_id  patient_id   enc_date       dept diag_code  cost_cad
      2           2 2023-02-20 Cardiology     I25.1    3200.0
     12          10 2023-11-03  Emergency     S52.5    2200.0
      7           7 2023-06-22 Cardiology       I10    2100.0
      1           1 2023-01-15  Emergency     J18.9    1850.0
     11         

---
## Filtering on derived expressions


In [11]:
# Filter on a computed expression — cannot use alias, write expression directly
print('=== High-cost per-dept summary, filtering on count > 1 ===')
q = """
SELECT dept,
       COUNT(*)          AS encounters,
       ROUND(AVG(cost_cad), 2) AS avg_cost,
       SUM(admitted)     AS admissions
FROM   encounters
GROUP BY dept
HAVING COUNT(*) > 1         -- HAVING filters after aggregation (like WHERE for groups)
ORDER BY avg_cost DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# WHERE on STRFTIME (date part)
print()
print('=== Encounters in second half of year (July–December) ===')
q2 = """
SELECT enc_id, enc_date, dept, cost_cad
FROM   encounters
WHERE  CAST(STRFTIME('%m', enc_date) AS INTEGER) >= 7
ORDER BY enc_date
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


=== High-cost per-dept summary, filtering on count > 1 ===
        dept  encounters  avg_cost  admissions
Orthopaedics           2    2975.0           1
  Cardiology           3    2350.0           3
   Emergency           4    1310.0           2
  Outpatient           5      99.0           0

=== Encounters in second half of year (July–December) ===
 enc_id   enc_date         dept  cost_cad
      8 2023-07-14   Outpatient      80.0
      9 2023-08-30    Emergency     410.0
     10 2023-09-12   Outpatient     110.0
     11 2023-10-01   Cardiology    1750.0
     12 2023-11-03    Emergency    2200.0
     13 2023-11-18   Outpatient      90.0
     14 2023-12-05 Orthopaedics     450.0


---
## Common Pitfalls

**1. `= NULL` never returns rows**
`WHERE diag_code = NULL` evaluates to NULL for every row, returning nothing. SQL uses three-valued logic — NULL comparisons produce NULL, not FALSE. Always use `IS NULL` and `IS NOT NULL`.

**2. NOT IN fails silently when the list contains a NULL**
`WHERE dept NOT IN ('Outpatient', NULL)` expands to `dept <> 'Outpatient' AND dept <> NULL`. The second condition is always NULL, so the entire AND is NULL — no rows pass. If the exclusion list comes from a subquery that might return NULLs, use `NOT EXISTS` instead.

**3. BETWEEN is inclusive on both ends**
`BETWEEN '2023-01-01' AND '2023-03-31'` includes all of March 31 up to `'2023-03-31 23:59:59'`. For datetime columns, use `>= '2023-01-01' AND < '2023-04-01'` to avoid boundary ambiguity.

**4. AND binds tighter than OR — always use parentheses when mixing**
`WHERE a = 1 OR b = 2 AND c = 3` is `WHERE a = 1 OR (b = 2 AND c = 3)`. This is legal SQL but frequently not what the author intended. Write explicit parentheses whenever you combine AND and OR.

**5. LIKE with a leading wildcard cannot use an index**
`WHERE diag_code LIKE '%18%'` forces a full table scan on even a well-indexed column. For suffix or substring searches at scale, use PostgreSQL full-text search (`tsvector`) or a trigram index (`pg_trgm`). The `LIKE 'J%'` prefix pattern can use a B-tree index normally.

**6. NULL rows are silently excluded — this can bias aggregates**
A `WHERE cost_cad > 0` filter that incidentally excludes NULL costs may make your average look higher than it is. Decide explicitly whether NULLs should be treated as zero, excluded, or flagged separately.


---
*sql_methods_library - Samantha McGarrigle*